In [3]:
%pip install ContrailOnlineCAClient

  Using cached ContrailOnlineCAClient-0.5.1-py3-none-any.whl.metadata (2.7 kB)
Using cached ContrailOnlineCAClient-0.5.1-py3-none-any.whl (197 kB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
from bs4 import BeautifulSoup
import os
import datetime

In [5]:
# Import third-party libraries
from cryptography import x509
from cryptography.hazmat.backends import default_backend
from contrail.security.onlineca.client import OnlineCaClient

In [6]:
CERTS_DIR = os.path.expanduser('~/.certs')
if not os.path.isdir(CERTS_DIR):
    os.makedirs(CERTS_DIR)

TRUSTROOTS_DIR = os.path.join(CERTS_DIR, 'ca-trustroots')
CREDENTIALS_FILE_PATH = os.path.join(CERTS_DIR, 'credentials.pem')

TRUSTROOTS_SERVICE = 'https://slcs.ceda.ac.uk/onlineca/trustroots/'
CERT_SERVICE = 'https://slcs.ceda.ac.uk/onlineca/certificate/'

In [7]:
def cert_is_valid(cert_file, min_lifetime=0):
    """
    Returns boolean - True if the certificate is in date.
    Optional argument min_lifetime is the number of seconds
    which must remain.

    :param cert_file: certificate file path.
    :param min_lifetime: minimum lifetime (seconds)
    :return: boolean
    """
    try:
        with open(cert_file, 'rb') as f:
            crt_data = f.read()
    except IOError:
        return False

    try:
        cert = x509.load_pem_x509_certificate(crt_data, default_backend())
    except ValueError:
        return False

    now = datetime.datetime.now()
    return (cert.not_valid_before <= now
            and cert.not_valid_after > now + datetime.timedelta(0, min_lifetime))

In [8]:
def setup_credentials():
    """
    Download and create required credentials files.

    Return True if credentials were set up.
    Return False if credentials were already set up.
    """
    if cert_is_valid(CREDENTIALS_FILE_PATH):
        print('[INFO] Security credentials already set up.')
        return False

    username = 'tnobrega'
    password = 'x9kp2HU1pX'

    if not username or not password:
        raise ValueError("CEDA_USERNAME and CEDA_PASSWORD environment variables are required")

    onlineca_client = OnlineCaClient()
    onlineca_client.ca_cert_dir = TRUSTROOTS_DIR

    trustroots = onlineca_client.get_trustroots(
        TRUSTROOTS_SERVICE,
        bootstrap=True,
        write_to_ca_cert_dir=True)

    key_pair, certs = onlineca_client.get_certificate(
        username,
        password,
        CERT_SERVICE,
        pem_out_filepath=CREDENTIALS_FILE_PATH)

    print('[INFO] Security credentials set up.')
    return True

In [10]:
def get_nc_file_links(directory_url):
    """
    Retrieve a list of NetCDF (.nc) file URLs from the given directory URL.

    :param directory_url: URL to the directory containing NetCDF files.
    :return: List of URLs to NetCDF files.
    """
    try:
        setup_credentials()
    except ValueError as e:
        print(e)
        return

    
    response = requests.get(directory_url, cert=(CREDENTIALS_FILE_PATH), verify=False)
    if response.status_code != 200:
        print(f"[ERROR] Failed to access {directory_url}")
        return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    nc_files = []
    
    for link in soup.find_all('a'):
        href = link.get('href')
        if href and href.endswith('.nc'):
            nc_files.append(directory_url.rstrip('/') + '/' + href)
    
    return nc_files

# Example usage
directory_url = 'https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/'
nc_file_links = get_nc_file_links(directory_url)
print(nc_file_links)

[INFO] Security credentials already set up.
['https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core-cloud-phy_faam_20120914_v500_r0_b731.nc', 'https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core_faam_20120914_v004_r0_b731.nc', 'https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core_faam_20120914_v004_r0_b731_1hz.nc', 'https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core_faam_20120914_v004_r1_b731.nc', 'https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core_faam_20120914_v004_r1_b731_1hz.nc', 'https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/faam-dropsonde_faam_20120914160228_r0_b731_proc.nc']


/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dap.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [11]:
def download_nc_file(file_url):
    """
    Download NetCDF file from the given URL.

    :param file_url: URL to the NetCDF file.
    """
    try:
        setup_credentials()
    except ValueError as e:
        print(e)
        return

    response = requests.get(file_url, cert=(CREDENTIALS_FILE_PATH), verify=False)
    filename = file_url.rsplit('/', 1)[-1]
    with open(filename, 'wb') as file_object:
        file_object.write(response.content)
    print(f"[INFO] File downloaded: {filename}")

In [14]:
url0 ='https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core-cloud-phy_faam_20120914_v500_r0_b731.nc'
url = nc_file_links[0]
print(url0)
print(url)

https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core-cloud-phy_faam_20120914_v500_r0_b731.nc
https://dap.ceda.ac.uk/badc/faam/data/2012/b731-sep-14/core_processed/core-cloud-phy_faam_20120914_v500_r0_b731.nc


In [15]:
download_nc_file(url)

[INFO] Security credentials already set up.


/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dap.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'auth.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'auth.ceda.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/srv/conda/envs/notebook/l

[INFO] File downloaded: core-cloud-phy_faam_20120914_v500_r0_b731.nc
